# MLOps Zoomcamp - Homework 2: Experiment Tracking with MLflow

In [ ]:
import os
import pickle
import mlflow
from mlflow.sklearn import autolog
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
import urllib.request

## Q1: Проверка версии MLflow

In [ ]:
import mlflow
print(f"MLflow version: {mlflow.__version__}")

## Q2: Загрузка и предобработка данных

In [ ]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
MONTHS = ["2023-01", "2023-02", "2023-03"]
DATA_DIR = "./data"

def download_file(month):
    filename = f"green_tripdata_{month}.parquet"
    url = f"{BASE_URL}/{filename}"
    dest = os.path.join(DATA_DIR, filename)
    if os.path.exists(dest):
        print(f"✓ {filename} уже существует")
        return True
    print(f"Скачивание {filename}...")
    urllib.request.urlretrieve(url, dest)
    print(f"✓ {filename} загружен")
    return True

os.makedirs(DATA_DIR, exist_ok=True)
for month in MONTHS:
    download_file(month)

In [ ]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df

def preprocess(df, dv, fit_dv=False):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    if fit_dv:
        X = dv.fit_transform(dicts)
    else:
        X = dv.transform(dicts)
    return X, dv

In [ ]:
df_train = read_dataframe(os.path.join(DATA_DIR, "green_tripdata_2023-01.parquet"))
df_val = read_dataframe(os.path.join(DATA_DIR, "green_tripdata_2023-02.parquet"))
df_test = read_dataframe(os.path.join(DATA_DIR, "green_tripdata_2023-03.parquet"))

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

dv = DictVectorizer()
X_train, dv = preprocess(df_train, dv, fit_dv=True)
X_val, _ = preprocess(df_val, dv, fit_dv=False)
X_test, _ = preprocess(df_test, dv, fit_dv=False)

# Сохранение
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "dv.pkl"), "wb") as f:
    pickle.dump(dv, f)
with open(os.path.join(OUTPUT_DIR, "train.pkl"), "wb") as f:
    pickle.dump((X_train, y_train), f)
with open(os.path.join(OUTPUT_DIR, "val.pkl"), "wb") as f:
    pickle.dump((X_val, y_val), f)
with open(os.path.join(OUTPUT_DIR, "test.pkl"), "wb") as f:
    pickle.dump((X_test, y_test), f)

print(f"✓ Сохранено 4 файла в {OUTPUT_DIR}")

## Q3: Обучение модели с MLflow Autolog

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("random-forest-training")
autolog()

with mlflow.start_run():
    rf = RandomForestRegressor(max_depth=10, random_state=0)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    print(f"RMSE на валидации: {rmse:.4f}")

## Q4: Запуск MLflow Tracking Server

Сервер нужно запустить в отдельном терминале:

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./artifacts --host 127.0.0.1 --port 5000
```

Затем открыть UI: http://localhost:5000

**Ответ Q4:** Нужно передать `--backend-store-uri` и `--default-artifact-root`

## Q5: Оптимизация гиперпараметров (HPO)

In [ ]:
mlflow.set_experiment("random-forest-hyperopt")

def objective(params):
    with mlflow.start_run():
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)
    return {'loss': rmse, 'status': STATUS_OK}

search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
    'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
    'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
    'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
    'random_state': 42
}

rstate = np.random.default_rng(42)
fmin(fn=objective, space=search_space, algo=tpe.suggest, max_evals=15, trials=Trials(), rstate=rstate)

## Q6: Регистрация лучшей модели

In [ ]:
HPO_EXPERIMENT_NAME = "random-forest-hyperopt"
EXPERIMENT_NAME = "random-forest-best-models"
RF_PARAMS = ['max_depth', 'n_estimators', 'min_samples_split', 'min_samples_leaf', 'random_state']

mlflow.set_experiment(EXPERIMENT_NAME)
autolog()

client = MlflowClient()
experiment = client.get_experiment_by_name(HPO_EXPERIMENT_NAME)
runs = client.search_runs(
    experiment_ids=experiment.experiment_id,
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

for run in runs:
    with mlflow.start_run():
        params = {k: int(v) for k, v in run.data.params.items() if k in RF_PARAMS}
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)
        val_rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        test_rmse = root_mean_squared_error(y_test, rf.predict(X_test))
        mlflow.log_metric("val_rmse", val_rmse)
        mlflow.log_metric("test_rmse", test_rmse)

# Найти лучшую модель
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
best_run = client.search_runs(
    experiment_ids=experiment.experiment_id,
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=1,
    order_by=["metrics.test_rmse ASC"]
)[0]

model_uri = f"runs:/{best_run.info.run_id}/model"
mlflow.register_model(model_uri, "random-forest-model")
print(f"✓ Модель зарегистрирована: {model_uri}")

## Итоговые результаты

### Структура файлов:
```
./data/                    # Скачанные данные
  - green_tripdata_2023-01.parquet
  - green_tripdata_2023-02.parquet
  - green_tripdata_2023-03.parquet
./output/                  # Предобработанные данные
  - dv.pkl
  - train.pkl
  - val.pkl
  - test.pkl
mlflow.db                  # БД MLflow tracking
./artifacts/               # Артефакты MLflow
```

### Где смотреть результаты в MLflow UI:
1. Откройте http://localhost:5000
2. Эксперименты:
   - `random-forest-training` - Q3: обучение базовой модели
   - `random-forest-hyperopt` - Q5: оптимизация гиперпараметров
   - `random-forest-best-models` - Q6: переобучение лучших моделей
3. Зарегистрированные модели: `random-forest-model`